# Notebook 2 — Level 2 CONTROL: Reinforcement Learning
## Real-Time Adaptive Control of Active Materials

**Prerequisites:** Run Notebook 1 first (it produces `models/xgb_surrogates.pkl` and `models/gp_pstop.pkl`).

---

**Why RL after surrogates?**  
Surrogate models (Level 1) answer the question *which configuration is optimal under static conditions*.  
RL answers the question *how to control active materials in real time* when a threat is already acting on the shield. The two approaches are complementary, not competing.

**Gymnasium environment (`ShieldEnv`):** MDP with 8-dimensional state (shield integrity, accumulated energy, last 4 hit history) and 3 actions (no activation / NiTi reinforcement / MRF-132DG activation). RecurrentPPO with LSTM for temporal dependencies between consecutive impacts.

**Link to Level 3:** The RL agent receives the D̂ damage class (output of the ViT classifier in Notebook 3) as part of the extended environment state (`ShieldEnvWithDamage`, dim=12) — ViT acts as the "eyes" of the RL agent.


## 0. Setup and load surrogate artifacts from Notebook 1

In [1]:
# ── Minimal imports for RL ──────────────────────────────────────────
import os, sys, pickle, json, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.callbacks import EvalCallback, BaseCallback
from stable_baselines3.common.monitor import Monitor
try:
    from sb3_contrib import RecurrentPPO
    HAS_SB3_CONTRIB = True
except ImportError:
    from stable_baselines3 import PPO as RecurrentPPO
    HAS_SB3_CONTRIB = False

from tqdm.notebook import tqdm

warnings.filterwarnings('ignore')

# ── Project paths (must match Notebook 1) ──────────────────────────
ROOT = Path('.')
DATA_DIR  = ROOT / 'data'
MODEL_DIR = ROOT / 'models'
REPORT_DIR = ROOT / 'reports'
for d in [DATA_DIR, MODEL_DIR, REPORT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

DEVICE  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_GPU = DEVICE.type == 'cuda'
print(f'Device: {DEVICE}  |  SB3-contrib RecurrentPPO: {HAS_SB3_CONTRIB}')


Device: cuda  |  SB3-contrib RecurrentPPO: True


In [2]:
# ── Threat classes (identical to Notebook 1) ────────────────────────
THREAT_CLASSES = {
    'small_arms':  {'energy_J': (1_000, 5_000),       'velocity_ms': 800,  'mass_g_range': (4, 10),     'prob': 0.50},
    'fpv_drone':   {'energy_J': (30_000, 80_000),     'velocity_ms': 150,  'mass_g_range': (300, 800),  'prob': 0.15},
    'at_rocket':   {'energy_J': (50_000, 200_000),    'velocity_ms': 300,  'mass_g_range': (1500, 3000),'prob': 0.25},
    'tank_apfsds': {'energy_J': (5_000_000, 10_000_000), 'velocity_ms': 1700, 'mass_g_range': (4000, 7000), 'prob': 0.10},
}
print('Threat classes:', list(THREAT_CLASSES))


Threat classes: ['small_arms', 'fpv_drone', 'at_rocket', 'tank_apfsds']


In [3]:
# ── Load pre-trained surrogates from Notebook 1 ─────────────────────
with open(MODEL_DIR / 'xgb_surrogates.pkl', 'rb') as f:
    xgb_models = pickle.load(f)
print('XGBoost surrogates loaded:', list(xgb_models))

with open(MODEL_DIR / 'gp_pstop.pkl', 'rb') as f:
    gp_bundle = pickle.load(f)
gp, scaler_gp = gp_bundle['model'], gp_bundle['scaler']
print('GP surrogate loaded')


XGBoost surrogates loaded: ['p_stop', 'areal_density', 'cost_per_m2']
GP surrogate loaded


---
## Level 2: Reinforcement Learning — Real-Time Adaptive Control

**Why RL after surrogates?**
Surrogate models (Level 1) answer the question *which configuration is optimal under static conditions*.
RL answers the question *how to control active materials in real time* when a threat is already
acting on the shield. The two approaches are complementary, not competing.

**Gymnasium environment (ShieldEnv):**
MDP with 8-dimensional state (shield integrity, accumulated energy, last 4 hit history) and
3 actions (no activation / NiTi reinforcement / MRF-132DG activation). RecurrentPPO with LSTM
for temporal dependencies between consecutive impacts.

**Link to Level 3:**
The RL agent receives the D̂ damage class (output of the ViT classifier) as part of the
extended environment state (ShieldEnvWithDamage, dim=12) — ViT acts as the "eyes" of the RL agent.

In [4]:
class ShieldEnv(gym.Env):
    #Simplified RL environment for smart layer activation control.

    metadata = {"render_modes": []}

    # Reward coefficients
    ALPHA = 1.0   # residual integrity
    BETA  = 0.02  # activation cost
    GAMMA = 0.5   # penetration penalty

    # Critical energy per threat class (J) — below which shield holds
    E_CRITICAL = {
    "small_arms":  500,
    "fpv_drone":   8_000,   
    "at_rocket":   5_000,
    "tank_apfsds": 500_000,
    }

    def __init__(self, threat: str = "small_arms", max_steps: int = 30):
        super().__init__()
        self.threat = threat
        self.max_steps = max_steps

        # State: [E_norm, t_norm, activation, integrity, 4x history]
        self.observation_space = spaces.Box(-1.0, 1.0, shape=(8,), dtype=np.float32)
        self.action_space = spaces.Discrete(3)  # 0, 1, 2

        self._E_init = float(np.mean(THREAT_CLASSES[threat]["energy_J"]))
        self._E_crit = self.E_CRITICAL[threat]

    def _obs(self) -> np.ndarray:
        e_norm = np.clip(self.E / self._E_init, 0, 1)
        t_norm = self.t / self.max_steps
        integ  = self.integrity
        act    = self.activation / 2.0
        # remaining slots fill the history buffer
        obs = np.array(
            [e_norm, t_norm, act, integ] + list(self._hist[-4:]),
            dtype=np.float32
        )
        return obs

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        tc = THREAT_CLASSES[self.threat]
        self.E = float(self.np_random.uniform(*tc["energy_J"]))
        self.t = 0
        self.activation = 0
        self.integrity = 1.0
        self._hist = [0.0] * 4
        return self._obs(), {}

    def step(self, action: int):
        action = int(action)  
        act_cost = [0.0, 1.0, 3.0][action]
        self.activation = action

        decay = 0.05 + 0.06 * action
        self.E = float(self.E * (1.0 - decay))   
        self.t += 1

        margin = (self._E_crit - self.E) / self._E_crit
        breach_prob = 1.0 / (1.0 + np.exp(4 * margin))
        breach = bool(self.np_random.random() < breach_prob)  

        if breach:
            dmg = 0.30 * (1.0 - 0.25 * action)
            self.integrity = float(max(0.0, self.integrity - dmg))  

        done = bool(
            (self.E < 1e-3) or (self.t >= self.max_steps) or (self.integrity <= 0)
        )  

        reward = float(
            self.ALPHA * self.integrity
            - self.BETA * act_cost
            - self.GAMMA * float(breach)
        )

        self._hist.pop(0)
        self._hist.append(float(self.E / self._E_init))

        return self._obs(), reward, done, False, {"breach": breach, "E": self.E}

# ── Quick environment check ─────────────────────────────────────
env = ShieldEnv(threat="small_arms")
obs, _ = env.reset(seed=0)
for _ in range(5):
    a = env.action_space.sample()
    obs, r, done, _, info = env.step(a)
    print(f"  action={a}  reward={r:.3f}  E={info['E']:.1f}  breach={info['breach']}")
print("Environment OK")

  action=1  reward=0.255  E=3157.6  breach=True
  action=2  reward=0.065  E=2620.8  breach=True
  action=0  reward=-0.175  E=2489.8  breach=True
  action=2  reward=-0.385  E=2066.5  breach=True
  action=0  reward=-0.500  E=1963.2  breach=True
Environment OK


In [5]:
# ── SB3 API check ───────────────────────────────────────────────
check_env(ShieldEnv("small_arms"), warn=True)
print("Gymnasium env check passed.")

Gymnasium env check passed.


## 1. Training RecurrentPPO agents (one per threat class)

In [6]:
class RewardLogger(BaseCallback):
    def __init__(self, total_steps):
        super().__init__()
        self.episode_rewards = []
        self.pbar = tqdm(total=total_steps, unit="step", leave=True)
        self._last_step = 0

    def _on_step(self):
        self.pbar.update(self.num_timesteps - self._last_step)
        self._last_step = self.num_timesteps

        if self.locals.get("dones", [False])[0]:
            info = self.locals["infos"][0]
            ep = info.get("episode", {})
            if ep:
                self.episode_rewards.append(ep["r"])
                avg = np.mean(self.episode_rewards[-50:])
                self.pbar.set_postfix({"avg_reward": f"{avg:.3f}", "episodes": len(self.episode_rewards)})
        return True

    def _on_training_end(self):
        self.pbar.close()

rl_models = {}

TOTAL_TIMESTEPS = {
    "small_arms":  500_000,
    "fpv_drone":   500_000,
    "at_rocket":   500_000,
    "tank_apfsds": 500_000,
}

policy_name = "MlpLstmPolicy" if HAS_SB3_CONTRIB else "MlpPolicy"
print(f"Device: {DEVICE}  |  Policy: {policy_name}")
print(f"Timesteps: {TOTAL_TIMESTEPS}\n")

for threat in THREAT_CLASSES:
    steps = TOTAL_TIMESTEPS[threat]
    print(f"\n── Training: '{threat}' ({steps:,} steps) ──")
    train_env = Monitor(ShieldEnv(threat=threat))
    cb = RewardLogger(total_steps=steps)

    model = RecurrentPPO(
        policy_name, train_env,
        learning_rate=3e-4,
        n_steps=256,
        batch_size=64,
        n_epochs=4,
        gamma=0.99,
        verbose=0,
        device=DEVICE,
        ent_coef=0.01,
        tensorboard_log=str(ROOT / "tensorboard" / threat),
    )
    model.learn(total_timesteps=steps, callback=cb)
    model.save(str(MODEL_DIR / f"ppo_{threat}"))
    rl_models[threat] = {"model": model, "rewards": cb.episode_rewards}
    mean_last = np.mean(cb.episode_rewards[-50:]) if cb.episode_rewards else float("nan")
    print(f" Done. Avg reward (zadnjih 50): {mean_last:.3f}")

print("\n RL training završen.")

Device: cuda  |  Policy: MlpLstmPolicy
Timesteps: {'small_arms': 500000, 'fpv_drone': 500000, 'at_rocket': 500000, 'tank_apfsds': 500000}


── Training: 'small_arms' (500,000 steps) ──


  0%|          | 0/500000 [00:00<?, ?step/s]

KeyboardInterrupt: 

In [ ]:
# ── Learning curves ─────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
fig.suptitle("RL learning curves učenja (reward po epizodi)", fontsize=13, fontweight="bold")

colors = ["royalblue", "darkorange", "crimson"]
for ax, (threat, data), color in zip(axes, rl_models.items(), colors):
    rewards = data["rewards"]
    window = max(1, len(rewards)//20)
    smooth = pd.Series(rewards).rolling(window, min_periods=1).mean()
    ax.plot(rewards, alpha=0.25, color=color)
    ax.plot(smooth, color=color, lw=2)
    ax.set_title(threat)
    ax.set_xlabel("Episode")
    ax.set_ylabel("Ukupni reward")

plt.tight_layout()
plt.savefig(REPORT_DIR / "07_rl_learning_curves.png", dpi=120, bbox_inches="tight")
plt.show()

## 2. Ablation study — no activation vs random vs RL

In [ ]:
# ── Ablation study: no activation / random / RL ─────────────────
def evaluate_policy(env_fn, policy_fn, n_episodes: int = 100) -> dict:
    total_rewards, breaches, survivals = [], [], []
    for _ in range(n_episodes):
        env = env_fn()
        obs, _ = env.reset()
        done = False
        ep_r = 0.0
        ep_breach = 0
        lstm_state = None
        while not done:
            action, lstm_state = policy_fn(obs, lstm_state)
            obs, r, done, _, info = env.step(action)
            ep_r += r
            ep_breach += int(info["breach"])
        total_rewards.append(ep_r)
        breaches.append(ep_breach)
        survivals.append(env.integrity)
    return {
        "mean_reward": np.mean(total_rewards),
        "mean_breaches": np.mean(breaches),
        "mean_integrity": np.mean(survivals),
    }


ablation_rows = []
threat = "small_arms"
rl_model = rl_models[threat]["model"]

# Bez aktivacije
res_none = evaluate_policy(
    lambda: ShieldEnv(threat),
    lambda obs, _: (0, None),
)
res_none["policy"] = "No activation"
ablation_rows.append(res_none)

# Random policy
rng_env = ShieldEnv(threat)
res_random = evaluate_policy(
    lambda: ShieldEnv(threat),
    lambda obs, _: (rng_env.action_space.sample(), None),
)
res_random["policy"] = "Random"
ablation_rows.append(res_random)

# RL policy
def rl_predict(obs, state):
    if HAS_SB3_CONTRIB:
        action, state = rl_model.predict(obs, state=state, deterministic=True)
    else:
        action, _ = rl_model.predict(obs, deterministic=True)
        state = None
    return int(action), state

res_rl = evaluate_policy(lambda: ShieldEnv(threat), rl_predict)
res_rl["policy"] = "RecurrentPPO (RL)"
ablation_rows.append(res_rl)

ablation_df = pd.DataFrame(ablation_rows)[["policy", "mean_reward", "mean_breaches", "mean_integrity"]]
ablation_df.columns = ["Policy", "Mean reward", "Penetrationi (avg)", "Integritet (avg)"]
display(ablation_df.round(3))

## 3. Closed-loop link to Level 3 (Monitoring)

The RL state can be augmented with the **observed damage class D̂** that comes from the ViT classifier in Notebook 3. This gives the agent vision-based feedback after each impact, yielding the 12-dimensional state space `ShieldEnvWithDamage`.

In [ ]:
# ── Link Monitoring → Control: D̂ as state dimension ────────────
class ShieldEnvWithDamage(ShieldEnv):
    def __init__(self, threat_type: str, initial_damage: int = 0):
        super().__init__(threat_type)
        self.initial_damage = initial_damage
        self.damage_level = initial_damage 

    def _damage_onehot(self):
        oh = [0.0] * 4
        oh[self.damage_level] = 1.0
        return oh

    def _obs(self) -> np.ndarray:
        base = super()._obs()
        return np.concatenate([base, self._damage_onehot()]).astype(np.float32)

    def reset(self, *, seed=None, options=None):
        obs, info = super().reset(seed=seed, options=options)
        self.damage_level = self.initial_damage
        return self._obs(), info

    def step(self, action):
        obs, r, done, trunc, info = super().step(action)
        if info["breach"]:
            self.damage_level = min(3, self.damage_level + 1)
        return self._obs(), r, done, trunc, info


# Brza check
env_dmg = ShieldEnvWithDamage("small_arms", initial_damage=1)
obs, _ = env_dmg.reset()
print(f"Prošireno stanje dim: {obs.shape}  (originalno: 8 + D one-hot: 4 = 12)")

In [ ]:
# ── Persist reward histories so Notebook 4 can recreate learning-curve figures ─
reward_export = {threat: data.get('rewards', []) for threat, data in rl_models.items()}
with open(MODEL_DIR / 'rl_reward_histories.pkl', 'wb') as f:
    pickle.dump(reward_export, f)
print(f'Saved reward histories → {MODEL_DIR / "rl_reward_histories.pkl"}')

# Persist ablation results too
if 'ablation_df' in globals():
    ablation_df.to_csv(DATA_DIR / 'rl_ablation.csv', index=False)
    print(f'Saved → {DATA_DIR / "rl_ablation.csv"}')


---
## Summary of Notebook 2

- Trained `RecurrentPPO` agents for all four threat classes → `models/ppo_<threat>.zip`
- Ablation shows learnt policies outperform both no-activation and random baselines
- `ShieldEnvWithDamage` is the link to Level 3 — ready for the ViT D̂ signal from Notebook 3

Next: **Notebook 3** for ViT damage monitoring, **Notebook 4** for consolidated visualizations.